In [1]:
# %% Raw data trust audit (BigQuery)
import pandas as pd
from IPython.display import display
from google.cloud import bigquery

PROJECT_ID = "voy-task"
RAW_DATASET = "lz_ae_task"

client = bigquery.Client(project=PROJECT_ID)

def bq_df(sql: str) -> pd.DataFrame:
    return client.query(sql).result().to_dataframe()

T_CUSTOMERS = f"`{PROJECT_ID}.{RAW_DATASET}.customers`"
T_ACQ = f"`{PROJECT_ID}.{RAW_DATASET}.acq_orders`"
T_ACTIVITY = f"`{PROJECT_ID}.{RAW_DATASET}.activity`"

print(f"Auditing raw tables in {PROJECT_ID}.{RAW_DATASET}")

# 1) Basic volume and key completeness
shape_and_keys = bq_df(
    f"""
    select 'customers' as table_name,
           count(*) as row_count,
           count(distinct cast(customer_id as string)) as distinct_customer_ids,
           countif(customer_id is null or trim(cast(customer_id as string)) = '') as null_or_blank_customer_id
    from {T_CUSTOMERS}
    union all
    select 'acq_orders' as table_name,
           count(*) as row_count,
           count(distinct cast(customer_id as string)) as distinct_customer_ids,
           countif(customer_id is null or trim(cast(customer_id as string)) = '') as null_or_blank_customer_id
    from {T_ACQ}
    union all
    select 'activity' as table_name,
           count(*) as row_count,
           count(distinct cast(customer_id as string)) as distinct_customer_ids,
           countif(customer_id is null or trim(cast(customer_id as string)) = '') as null_or_blank_customer_id
    from {T_ACTIVITY}
    """
)
display(shape_and_keys)

# 2) Duplicates and key-level granularity
dupes = bq_df(
    f"""
    with customer_dupes as (
        select count(*) as duplicate_customer_keys
        from (
            select cast(customer_id as string) as customer_id, count(*) as n
            from {T_CUSTOMERS}
            group by 1
            having n > 1
        )
    ),
    acq_multi as (
        select count(*) as customers_with_multiple_acq_rows
        from (
            select cast(customer_id as string) as customer_id, count(*) as n
            from {T_ACQ}
            group by 1
            having n > 1
        )
    )
    select * from customer_dupes cross join acq_multi
    """
)
display(dupes)

# 3) Referential integrity (orphans)
orphans = bq_df(
    f"""
    with cust as (
        select distinct cast(customer_id as string) as customer_id
        from {T_CUSTOMERS}
    )
    select
        (select count(*) from {T_ACQ} a left join cust c on cast(a.customer_id as string) = c.customer_id
          where c.customer_id is null and a.customer_id is not null and trim(cast(a.customer_id as string)) != '') as acq_orphan_rows,
        (select count(*) from {T_ACTIVITY} x left join cust c on cast(x.customer_id as string) = c.customer_id
          where c.customer_id is null and x.customer_id is not null and trim(cast(x.customer_id as string)) != '') as activity_orphan_rows
    """
)
display(orphans)

# 4) Activity interval validity
interval_quality = bq_df(
    f"""
    select
        count(*) as total_activity_rows,
        countif(subscription_id is null or trim(cast(subscription_id as string)) = '') as null_or_blank_subscription_id,
        countif(from_date is null) as null_from_date,
        countif(to_date is not null and to_date < from_date) as inverted_intervals,
        countif(to_date is not null and to_date = from_date) as zero_duration_intervals,
        countif(to_date is null) as open_intervals
    from {T_ACTIVITY}
    """
)
display(interval_quality)

# 5) Categorical hygiene checks (whitespace/case consistency signals)
categorical_hygiene = bq_df(
    f"""
    select
        (select countif(customer_country is not null and customer_country != trim(customer_country)) from {T_CUSTOMERS}) as country_with_outer_whitespace,
        (select count(distinct customer_country) from {T_CUSTOMERS} where customer_country is not null) as distinct_country_values,
        (select countif(taxonomy_business_category_group is not null and taxonomy_business_category_group != trim(taxonomy_business_category_group)) from {T_ACQ}) as category_with_outer_whitespace,
        (select count(distinct taxonomy_business_category_group) from {T_ACQ} where taxonomy_business_category_group is not null) as distinct_category_values
    """
)
display(categorical_hygiene)

# 6) PASS/FAIL audit summary (what must be fixed before trusted modeling)
critical_checks = [
    ("customers null/blank customer_id", int(shape_and_keys.loc[shape_and_keys.table_name == 'customers', 'null_or_blank_customer_id'].iloc[0])),
    ("acq_orders null/blank customer_id", int(shape_and_keys.loc[shape_and_keys.table_name == 'acq_orders', 'null_or_blank_customer_id'].iloc[0])),
    ("activity null/blank customer_id", int(shape_and_keys.loc[shape_and_keys.table_name == 'activity', 'null_or_blank_customer_id'].iloc[0])),
    ("customers duplicate customer_id keys", int(dupes['duplicate_customer_keys'].iloc[0])),
    ("acq_orders orphan rows (customer missing in customers)", int(orphans['acq_orphan_rows'].iloc[0])),
    ("activity orphan rows (customer missing in customers)", int(orphans['activity_orphan_rows'].iloc[0])),
    ("activity null/blank subscription_id", int(interval_quality['null_or_blank_subscription_id'].iloc[0])),
    ("activity null from_date", int(interval_quality['null_from_date'].iloc[0])),
    ("activity inverted intervals (to_date < from_date)", int(interval_quality['inverted_intervals'].iloc[0])),
]

summary = pd.DataFrame(
    [{"check": name, "count": count, "status": "FAIL" if count > 0 else "PASS"} for name, count in critical_checks]
)
display(summary)

overall_status = "FAIL" if (summary["status"] == "FAIL").any() else "PASS"
print("\nOverall raw-data trust status:", overall_status)
if overall_status == "FAIL":
    print("Action: clean and quarantine failing records before relying on downstream models.")
else:
    print("Action: raw data passes critical checks and is suitable for trusted modeling.")

# 7) Optional: show top failing examples for triage
if int(interval_quality['inverted_intervals'].iloc[0]) > 0:
    print("\nSample inverted intervals:")
    display(
        bq_df(
            f"""
            select customer_id, subscription_id, from_date, to_date
            from {T_ACTIVITY}
            where to_date is not null and to_date < from_date
            limit 50
            """
        )
    )


Auditing raw tables in voy-task.lz_ae_task


c:\Users\jcstr\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,table_name,row_count,distinct_customer_ids,null_or_blank_customer_id
0,customers,532848,532848,0
1,activity,2176168,512366,0
2,acq_orders,508694,508694,0


c:\Users\jcstr\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,duplicate_customer_keys,customers_with_multiple_acq_rows
0,0,0


...

,total_activity_rows,null_or_blank_subscription_id,null_from_date,inverted_intervals,zero_duration_intervals,open_intervals
0,2176168,0,0,0,171983,0


c:\Users\jcstr\AppData\Local\Programs\Python\Python311\Lib\site-packages\google\cloud\bigquery\table.py:2082: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,country_with_outer_whitespace,distinct_country_values,category_with_outer_whitespace,distinct_category_values
0,0,2,0,7


,check,count,status
0,customers null/blank customer_id,0,PASS
1,acq_orders null/blank customer_id,0,PASS
2,activity null/blank customer_id,0,PASS
3,customers duplicate customer_id keys,0,PASS
4,acq_orders orphan rows (customer missing in cu...,0,PASS
5,activity orphan rows (customer missing in cust...,0,PASS
6,activity null/blank subscription_id,0,PASS
7,activity null from_date,0,PASS
8,activity inverted intervals (to_date < from_date),0,PASS



Overall raw-data trust status: PASS
Action: raw data passes critical checks and is suitable for trusted modeling.
